In [1]:
import torch
import torch.nn as nn
from torch import tensor

In [17]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=1,
            bias=False,
            batch_first=True
        )  
        self.rnn.load_state_dict({
            "weight_ih_l0": torch.tensor([
                [0.3]
            ]),
            "weight_hh_l0": torch.tensor([
                [0.4]
            ])
        })
        
    def forward(self, x):
        output, hidden = self.rnn(x)
        hidden = hidden[-1]
        return hidden

In [18]:
model = MyModel()

In [22]:
x = torch.tensor([
    [
        [0.1], [0.2], [0.3], [0.4],
    ]
])
x.shape

torch.Size([1, 4, 1])

In [23]:
y_pred = model(x)
y_pred

tensor([[0.1657]], grad_fn=<SelectBackward0>)

In [24]:
y_target = tensor([[0.5]])

In [25]:
loss_fn = nn.MSELoss()
loss = loss_fn(y_pred, y_target)
loss

tensor(0.1117, grad_fn=<MseLossBackward0>)

In [26]:
loss.backward()

In [27]:
model.rnn.weight_ih_l0.grad

tensor([[-0.3615]])

In [28]:
model.rnn.weight_hh_l0.grad

tensor([[-0.0983]])

In [30]:
W_ih = model.rnn.weight_ih_l0.item()
W_hh = model.rnn.weight_hh_l0.item()

print(W_ih)
print(W_hh)

0.30000001192092896
0.4000000059604645


In [38]:
h0 = 0
x1 = 0.1
x2 = 0.2
x3 = 0.3
x4 = 0.4

z1 = torch.tensor(x1 * W_ih + h0 * W_hh)
h1 = torch.tanh(z1)

z2 = torch.tensor(x2 * W_ih + h1.item() * W_hh)
h2 = torch.tanh(z2)

z3 = torch.tensor(x3 * W_ih + h2.item() * W_hh)
h3 = torch.tanh(z3)

z4 = torch.tensor(x4 * W_ih + h3.item() * W_hh)
h4 = torch.tanh(z4)

y_pred = h4.item()
y_target = 0.5

L = (y_pred - y_target) ** 2
dL_dypred = 2 * (y_pred - y_target)
dL_dh4 = dL_dypred

dh4_dz4 = 1 - h4.item() ** 2
dz4_dW_ih = x4

dh3_dz3 = 1 - h3.item() ** 2
dh2_dz2 = 1 - h2.item() ** 2
dh1_dz1 = 1 - h1.item() ** 2

dz3_dW_ih = x3
dz2_dW_ih = x2
dz1_dW_ih = x1

dz4_dh3 = W_hh
dz3_dh2 = W_hh
dz2_dh1 = W_hh

dh1_dW_ih = dh1_dz1 * (dz1_dW_ih + 0)
dh2_dW_ih = dh2_dz2 * (dz2_dW_ih + dz2_dh1 * dh1_dW_ih)
dh3_dW_ih = dh3_dz3 * (dz3_dW_ih + dz3_dh2 * dh2_dW_ih)
dL_dW_ih = dL_dh4 * dh4_dz4 * (dz4_dW_ih + dz4_dh3 * dh3_dW_ih)
dL_dW_ih

-0.36148407892088286

In [39]:
model.rnn.weight_ih_l0.grad.item()

-0.3614840805530548

In [42]:
dz4_dW_hh = h3.item()
dz3_dW_hh = h2.item()
dz2_dW_hh = h1.item()
dz1_dW_hh = 0

dh1_dW_hh = dh1_dz1 * (dz1_dW_hh + 0)
dh2_dW_hh = dh2_dz2 * (dz2_dW_hh + dz2_dh1 * dh1_dW_hh)
dh3_dW_hh = dh3_dz3 * (dz3_dW_hh + dz3_dh2 * dh2_dW_hh)
dL_dW_hh = dL_dh4 * dh4_dz4 * (dz4_dW_hh + dz4_dh3 * dh3_dW_hh)
dL_dW_hh

-0.0983367169058599

In [43]:
model.rnn.weight_hh_l0.grad.item()

-0.09833671897649765

In [ ]:
# y = tanh(x) -> dy/dx = 1 - tanh(x)^2 = 1 - y^2